# 09 — Multi-VSL RGB baseline (50 classes)

This standalone Colab notebook prepares an **M-VSL200 center-view** subset and trains the frozen VideoMAE V2 + compact RGB Transformer baseline. It preserves the official signer-disjoint train/validation/test split. Source videos and the Git checkout stay in the temporary Colab runtime; only manifests, logs, checkpoints, predictions, and figures are written to Google Drive.

Class selection uses official **training clip counts only**. Validation chooses the checkpoint and triggers early stopping. Test remains disabled until the configuration is frozen.

In [ ]:
PROJECT_GIT_REF = 'feat/multi-vsl-baseline'
OFFICIAL_REPOSITORY = 'https://github.com/Etdihatthoc/Multi-VSL_WACV_2025.git'
OFFICIAL_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1yUU1m2hy_CjaXDDoR_6i9Y3T1XL2pD4C?usp=sharing'
CLASS_COUNT = 50
MAX_EPOCHS = 60              # ceiling; early stopping normally finishes earlier
BATCH_SIZE = 2
RGB_EMBEDDING_DIM = 64
RGB_LAYERS = 1
RGB_HEADS = 2
RGB_DROPOUT = 0.50
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.04
LABEL_SMOOTHING = 0.15
EARLY_STOPPING_PATIENCE = 6
EARLY_STOPPING_MIN_DELTA = 0.005
RUN_TEST = False             # change once, only after the model is frozen
RESUME = True
DOWNLOAD_OFFICIAL_FOLDER = True
# If the public folder downloader is rate-limited, upload/extract the same official
# files under /content/multi_vsl_runtime/videos and set this to False.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
RUNTIME_ROOT = Path('/content/multi_vsl_runtime')
PROJECT_ROOT = Path('/content/silent-signal')
METADATA_REPO = RUNTIME_ROOT / 'Multi-VSL_WACV_2025'
VIDEO_ROOT = RUNTIME_ROOT / 'videos'
RESULTS_ROOT = Path('/content/drive/MyDrive/silent-signal-results/multi-vsl/mvsl200_center_top50')
PREPARED_ROOT = RESULTS_ROOT / 'prepared'
RUN_ROOT = RESULTS_ROOT / 'baseline_videomaev2_rgb_compact64_v1'
for path in (RUNTIME_ROOT, VIDEO_ROOT, PREPARED_ROOT, RUN_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print('Temporary videos:', VIDEO_ROOT)
print('Persistent outputs only:', RESULTS_ROOT)


In [ ]:
import subprocess, sys, time

def run(command, cwd=None):
    command = [str(value) for value in command]
    print('+', ' '.join(command), flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    print(f'Finished in {(time.perf_counter() - started) / 60:.1f} min', flush=True)
    if code:
        raise subprocess.CalledProcessError(code, command)

if not PROJECT_ROOT.exists():
    run(['git', 'clone', '--depth', '1', '--branch', PROJECT_GIT_REF, 'https://github.com/stillthethrone/silent-signal.git', PROJECT_ROOT])
if not METADATA_REPO.exists():
    run(['git', 'clone', '--depth', '1', OFFICIAL_REPOSITORY, METADATA_REPO])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{PROJECT_ROOT}[training]', 'opencv-python-headless', 'matplotlib', 'transformers', 'gdown'])


## Download videos to temporary runtime

The public Google Drive folder is maintained by the Multi-VSL authors. Google may rate-limit large anonymous folder downloads. This cell never copies source videos into your Drive. The preparation step below verifies that every official clip required by the selected classes exists, so an incomplete download cannot silently start training.

In [ ]:
if DOWNLOAD_OFFICIAL_FOLDER:
    run([sys.executable, '-m', 'gdown', '--folder', '--remaining-ok', OFFICIAL_DRIVE_FOLDER_URL, '-O', VIDEO_ROOT])
video_count = sum(1 for _ in VIDEO_ROOT.rglob('*.mp4'))
print(f'Runtime MP4 files: {video_count:,}')
if video_count == 0:
    raise RuntimeError('No videos found. Put the official MP4 files under /content/multi_vsl_runtime/videos.')


## Prepare and verify the official split

The official M-VSL200 center-view CSV files are used unchanged. The command checks filename membership, class coverage, duplicate samples, and signer isolation before writing the small reproducibility files to Drive.

In [ ]:
run([sys.executable, '-m', 'silent_signal.cli.prepare_multi_vsl_demo',
     '--metadata-root', METADATA_REPO / 'data' / 'label_1_200',
     '--video-root', VIDEO_ROOT,
     '--output-root', PREPARED_ROOT,
     '--classes', CLASS_COUNT])
MANIFEST = PREPARED_ROOT / 'multi_vsl_200_center_top50.csv'
SELECTION = PREPARED_ROOT / 'multi_vsl_200_center_top50_selection.json'


In [ ]:
import json, torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then run all again.')
print('GPU:', torch.cuda.get_device_name(0))
summary = json.loads((PREPARED_ROOT / 'multi_vsl_200_center_top50_summary.json').read_text())
print(json.dumps(summary, indent=2, ensure_ascii=False))


## Train with validation early stopping

VideoMAE V2 is frozen. Only the compact 64-dimensional RGB Transformer and classifier are learned. Checkpoints are resumable from Drive. `RUN_TEST=False` protects the held-out test set during model development.

In [ ]:
command = [sys.executable, '-u', '-m', 'silent_signal.cli.train_videomaev2_demo',
    '--manifest', MANIFEST, '--selection-report', SELECTION,
    '--dataset-root', VIDEO_ROOT, '--output-root', RUN_ROOT,
    '--classes', CLASS_COUNT, '--epochs', MAX_EPOCHS, '--batch-size', BATCH_SIZE,
    '--learning-rate', LEARNING_RATE, '--weight-decay', WEIGHT_DECAY,
    '--label-smoothing', LABEL_SMOOTHING,
    '--rgb-embedding-dim', RGB_EMBEDDING_DIM, '--rgb-layers', RGB_LAYERS,
    '--rgb-heads', RGB_HEADS, '--rgb-dropout', RGB_DROPOUT,
    '--early-stopping-patience', EARLY_STOPPING_PATIENCE,
    '--early-stopping-min-delta', EARLY_STOPPING_MIN_DELTA,
    '--checkpoint-every', 20, '--progress-every', 5, '--device', 'cuda']
if RESUME:
    command.append('--resume')
if RUN_TEST:
    command.append('--run-test')
run(command, cwd=PROJECT_ROOT)


In [ ]:
from IPython.display import Image, display
report = json.loads((RUN_ROOT / 'baseline_report.json').read_text())
print(json.dumps({key: report[key] for key in ('completed_epochs', 'best_epoch', 'best_validation_loss', 'early_stopping', 'evaluation')}, indent=2))
display(Image(filename=str(RUN_ROOT / 'training_curves.png')))
print('Saved outputs:', RUN_ROOT)
print('Source videos remain temporary:', VIDEO_ROOT)
